# Multivariate EDA — NZ Funding vs Australia Funding

This notebook compares **New Zealand** and **Australian** government funding rates for undergraduate degrees across broad fields of study.

**Data sources:**
- NZ: `data/raw/NZ Funding/NZ_TEC_SAC_rates_all_years.csv` — TEC SAC government subsidy per EFTS (Level 2, excl. GST), 2016–2025
- AUS: `data/clean/AnnualFundingAUS2019-2026_category_summary.csv` — avg Commonwealth Contribution per programme (AUD, excl. GST), 2019–2026

**Conceptual equivalence:**

| NZ | AUS |
|---|---|
| TEC SAC rate (NZD per EFTS, excl. GST) | `AvgCommonwealthContribution` (AUD per programme slot, excl. GST) |
| 19 cost categories (A–W) | 11 `CategoryKey` groups |
| Annual TEC review cycle | Fixed by legislation; JRG repriced in 2021 |

**Overlapping years:** 2019–2025 (NZ has 2016–2025; AUS has 2019–2026).

**Role in the JRGS project:** Documenting the funding parallel-trends baseline establishes whether any AUS enrolment divergence from NZ after 2021 is driven by the funding channel (JRG price shock) or by other factors.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')

# Resolve data paths
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
nz_sac_path = aus_fund_path = None

for root in candidate_roots:
    c_nz  = root / 'data' / 'raw' / 'NZ Funding' / 'NZ_TEC_SAC_rates_all_years.csv'
    c_aus = root / 'data' / 'clean' / 'AnnualFundingAUS2019-2026_category_summary.csv'
    if c_nz.exists() and c_aus.exists():
        nz_sac_path, aus_fund_path = c_nz, c_aus
        break

if nz_sac_path is None or aus_fund_path is None:
    raise FileNotFoundError(f'One or both funding datasets not found from: {Path.cwd()}')

df_nz_sac  = pd.read_csv(nz_sac_path)
df_aus_fund = pd.read_csv(aus_fund_path)

print('=== NZ TEC SAC Rates ===')
print(f'Shape: {df_nz_sac.shape}  | years: {sorted(df_nz_sac["year"].unique())} | categories: {sorted(df_nz_sac["category_code"].unique())}')
display(df_nz_sac.head())

print('\n=== AUS Commonwealth Contributions ===')
print(f'Shape: {df_aus_fund.shape}  | years: {sorted(df_aus_fund["Year"].unique())}')
print('Columns:', df_aus_fund.columns.tolist())
display(df_aus_fund.head())

## 1. Harmonize and Merge the Funding Datasets

**NZ:** Map TEC cost categories (letter codes) to shared `category_key` integers, then aggregate the SAC rate per `(category_key, year)`. Convert NZD → AUD using a fixed approximate rate (0.91).

**AUS:** Rename `Year` → `year`, `CategoryKey` → `category_key`. Use `AvgCommonwealthContribution` as the per-student government subsidy metric.

**Overlap:** Restrict to years 2019–2025 (present in both datasets).

In [ ]:
# ── NZ: map TEC category_code → category_key ─────────────────────────────────
NZ_CAT_TO_KEY = {
    'V': 1,   # Science           → Natural & Physical Sciences
    'C': 2,   # Computing/IT      → Information Technology
    'N': 3,   # Priority Eng      → Engineering & Related Tech
    'H': 5,   # Agriculture/Env   → Agriculture, Environmental & Related
    'B': 6,   # Health sciences   → Health
    'I': 7,   # Teaching          → Education
    'J': 8,   # Business/Commerce → Management & Commerce
    'A': 9,   # Arts/Humanities   → Society & Culture
}

NZD_TO_AUD = 0.91  # approximate long-run average

nz_mapped = df_nz_sac[df_nz_sac['category_code'].isin(NZ_CAT_TO_KEY)].copy()
nz_mapped['category_key'] = nz_mapped['category_code'].map(NZ_CAT_TO_KEY)

# Average SAC rate per (category_key, year) — some keys receive multiple TEC categories
nz_by_key = (nz_mapped
             .groupby(['category_key', 'year'], as_index=False)['sac_rate_level2_excl_gst']
             .mean()
             .rename(columns={'sac_rate_level2_excl_gst': 'NZ_SAC_NZD'}))
nz_by_key['NZ_SAC_AUD'] = (nz_by_key['NZ_SAC_NZD'] * NZD_TO_AUD).round(0)

# ── AUS: rename and clean ─────────────────────────────────────────────────────
aus_clean = (df_aus_fund[df_aus_fund['CategoryKey'] != 99]   # drop aggregate total row
             .rename(columns={
                 'Year':                       'year',
                 'CategoryKey':                'category_key',
                 'Category':                   'field_of_study',
                 'AvgCommonwealthContribution': 'AUS_CCont_AUD',
                 'AvgMaximumStudentContribution': 'AUS_StudentFee_AUD',
             })
             [['year', 'category_key', 'field_of_study', 'AUS_CCont_AUD', 'AUS_StudentFee_AUD']])

# ── Merge on (category_key, year) — keep only common keys and overlap years ───
common_keys  = set(nz_by_key['category_key'].unique()) & set(aus_clean['category_key'].unique())
nz_years     = set(nz_by_key['year'].unique())
aus_years    = set(aus_clean['year'].unique())
common_years = sorted(nz_years & aus_years)

merged = (nz_by_key[nz_by_key['category_key'].isin(common_keys) & nz_by_key['year'].isin(common_years)]
          .merge(aus_clean[aus_clean['category_key'].isin(common_keys) & aus_clean['year'].isin(common_years)],
                 on=['category_key', 'year'], how='inner')
          .sort_values(['category_key', 'year'])
          .reset_index(drop=True))

merged['Fund_Gap_AUD'] = merged['AUS_CCont_AUD'] - merged['NZ_SAC_AUD']

print(f'Shared category keys ({len(common_keys)}): {sorted(common_keys)}')
print(f'Overlapping years: {common_years}')
print(f'Merged shape: {merged.shape}')
print('\nMissing values:')
print(merged.isna().sum())
display(merged.head(16))

## 2. Inspect the Merged Funding Dataset

Coverage, summary statistics, and scale comparison between NZ and AUS.

In [ ]:
print('Merged dataset info:')
merged.info()

print('\nSummary statistics:')
display(merged[['NZ_SAC_NZD', 'NZ_SAC_AUD', 'AUS_CCont_AUD', 'Fund_Gap_AUD']].describe().round(0))

# Snapshot comparison in most recent common year
latest = max(common_years)
snap = (merged[merged['year'] == latest]
        [['field_of_study', 'category_key', 'NZ_SAC_AUD', 'AUS_CCont_AUD', 'Fund_Gap_AUD']]
        .sort_values('category_key'))
print(f'\nFunding comparison in {latest} (AUD):')
display(snap)

# Overall averages
print(f'\nAverage NZ SAC (AUD): {merged["NZ_SAC_AUD"].mean():,.0f}')
print(f'Average AUS CCont:    {merged["AUS_CCont_AUD"].mean():,.0f}')
print(f'Average gap (AUS-NZ): {merged["Fund_Gap_AUD"].mean():,.0f}')

## 3. Compare NZ and AUS Funding Levels Over Time

Raw and indexed funding rate trends by field, with 2021 JRG discontinuity highlighted.

In [ ]:
# Long format for seaborn
fund_long = merged.melt(
    id_vars=['category_key', 'field_of_study', 'year'],
    value_vars=['NZ_SAC_AUD', 'AUS_CCont_AUD'],
    var_name='Country', value_name='Rate_AUD'
)
fund_long['Country'] = fund_long['Country'].map(
    {'NZ_SAC_AUD': 'New Zealand', 'AUS_CCont_AUD': 'Australia'})

# --- Total / aggregate trend ---
agg = fund_long.groupby(['year', 'Country'], as_index=False)['Rate_AUD'].mean()
plt.figure(figsize=(10, 5))
sns.lineplot(data=agg, x='year', y='Rate_AUD', hue='Country', marker='o', linewidth=2.5)
plt.axvline(2021, color='red', ls='--', lw=1.5, label='JRG (AUS 2021)')
plt.title('Average Government Subsidy per Student: NZ vs AUS (AUD equivalent)')
plt.ylabel('AUD per EFTS / EFTSL')
plt.legend()
plt.tight_layout()
plt.show()

# --- Facet by field ---
g = sns.relplot(
    data=fund_long, x='year', y='Rate_AUD', hue='Country',
    col='field_of_study', col_wrap=3, kind='line', marker='o',
    facet_kws={'sharey': False, 'sharex': True}, height=3.2, aspect=1.2
)
g.set_titles('{col_name}')
for ax in g.axes.flat:
    ax.axvline(2021, color='red', ls='--', lw=1)
    ax.tick_params(axis='x', rotation=45)
g.fig.suptitle('NZ vs AUS Government Subsidy by Field (AUD) — Red = JRG 2021', y=1.02)
plt.show()

# --- Indexed: base 2019 = 100 ---
BASE = 2019
base_vals = (fund_long[fund_long['year'] == BASE]
             .set_index(['field_of_study', 'Country'])['Rate_AUD'])
fund_long['Index'] = fund_long.apply(
    lambda r: r['Rate_AUD'] / base_vals.get((r['field_of_study'], r['Country']), np.nan) * 100,
    axis=1
)

g2 = sns.relplot(
    data=fund_long.dropna(subset=['Index']),
    x='year', y='Index', hue='Country',
    col='field_of_study', col_wrap=3, kind='line', marker='o',
    facet_kws={'sharey': False, 'sharex': True}, height=3.2, aspect=1.2
)
g2.set_titles('{col_name}')
for ax in g2.axes.flat:
    ax.axvline(2021, color='red', ls='--', lw=1)
    ax.axhline(100, color='grey', ls=':', lw=0.8)
    ax.tick_params(axis='x', rotation=45)
g2.fig.suptitle(f'NZ vs AUS Funding Indexed ({BASE}=100) — Red = JRG 2021', y=1.02)
plt.show()

## 4. Parallel Trends Assessment for Funding

Before 2021, NZ and AUS funding should show **parallel trends** if the control group is valid. A 2021 discontinuity in AUS (JRG) that is absent in NZ isolates the policy's funding channel.

In [ ]:
from scipy import stats as scipy_stats

PRE_YEARS  = [y for y in common_years if y < 2021]
POST_YEARS = [y for y in common_years if y >= 2021]

# YoY growth rates per field and country
merged_s = merged.sort_values(['category_key', 'year']).copy()
merged_s['NZ_yoy_%']  = merged_s.groupby('category_key')['NZ_SAC_AUD'].pct_change() * 100
merged_s['AUS_yoy_%'] = merged_s.groupby('category_key')['AUS_CCont_AUD'].pct_change() * 100

# AUS funding starts 2019 → PRE_YEARS = [2019, 2020]. After pct_change(), 2019
# is NaN (no prior year), leaving only 1 YoY observation per field (year 2020).
# Pearson r requires ≥2 — show the 2019→2020 directional comparison instead.
pre_s = merged_s[merged_s['year'].isin(PRE_YEARS)].dropna(subset=['NZ_yoy_%', 'AUS_yoy_%'])
print('NOTE: AUS funding data starts 2019 → only 1 pre-treatment YoY observation per field')
print('(2019→2020). Pearson r requires ≥2 — showing directional comparison table instead.\n')
print('Pre-treatment YoY growth rates (2019→2020) by field:')
yoy_table = pre_s[['field_of_study', 'category_key', 'NZ_yoy_%', 'AUS_yoy_%']].copy()
yoy_table['Direction_Match'] = np.sign(yoy_table['NZ_yoy_%']) == np.sign(yoy_table['AUS_yoy_%'])
display(yoy_table.sort_values('category_key'))
n_match = int(yoy_table['Direction_Match'].sum())
print(f'\n{n_match}/{len(yoy_table)} fields had funding moving in the same direction in 2019→2020.')

# Pre vs post DiD on funding: (AUS post - AUS pre) - (NZ post - NZ pre)
avg_pre_nz   = merged_s[merged_s['year'].isin(PRE_YEARS)]['NZ_SAC_AUD'].mean()
avg_post_nz  = merged_s[merged_s['year'].isin(POST_YEARS)]['NZ_SAC_AUD'].mean()
avg_pre_aus  = merged_s[merged_s['year'].isin(PRE_YEARS)]['AUS_CCont_AUD'].mean()
avg_post_aus = merged_s[merged_s['year'].isin(POST_YEARS)]['AUS_CCont_AUD'].mean()

did_fund = (avg_post_aus - avg_pre_aus) - (avg_post_nz - avg_pre_nz)
print(f'\nPre-period avg NZ SAC: {avg_pre_nz:,.0f} AUD  |  Post: {avg_post_nz:,.0f} AUD  |  Δ={avg_post_nz-avg_pre_nz:+,.0f}')
print(f'Pre-period avg AUS CCont: {avg_pre_aus:,.0f} AUD  |  Post: {avg_post_aus:,.0f} AUD  |  Δ={avg_post_aus-avg_pre_aus:+,.0f}')
print(f'\nFunding DiD estimate (AUS Δ − NZ Δ): {did_fund:+,.0f} AUD')
if did_fund > 500:
    print('  → AUS government subsidy rose substantially more than NZ post-2021 (STEM/priority fields gained).')
elif did_fund < -500:
    print('  → AUS government subsidy fell relative to NZ post-2021 (non-priority fields lost funding).')
else:
    print('  → Modest difference: AUS and NZ funding evolved at roughly similar rates on average.')

# Per-field DiD
field_did = []
for key, g in merged_s.groupby('category_key'):
    field = g['field_of_study'].iloc[0]
    pre_nz  = g[g['year'].isin(PRE_YEARS)]['NZ_SAC_AUD'].mean()
    post_nz = g[g['year'].isin(POST_YEARS)]['NZ_SAC_AUD'].mean()
    pre_aus  = g[g['year'].isin(PRE_YEARS)]['AUS_CCont_AUD'].mean()
    post_aus = g[g['year'].isin(POST_YEARS)]['AUS_CCont_AUD'].mean()
    field_did.append({
        'field_of_study': field,
        'category_key': key,
        'NZ_delta': post_nz - pre_nz,
        'AUS_delta': post_aus - pre_aus,
        'DiD_AUD': (post_aus - pre_aus) - (post_nz - pre_nz),
    })
field_did_df = pd.DataFrame(field_did).sort_values('DiD_AUD', ascending=False)
print('\nPer-field funding DiD (AUS Δ − NZ Δ, AUD):')
display(field_did_df)

## 5. Multivariate Relationship Analysis

Scatter, correlation, and heatmap of the NZ vs AUS funding gap across fields and years.

In [ ]:
# Scatter: NZ_SAC_AUD vs AUS_CCont_AUD
plt.figure(figsize=(9, 7))
sns.scatterplot(data=merged, x='NZ_SAC_AUD', y='AUS_CCont_AUD',
                hue='field_of_study', style='year', s=90)
sns.regplot(data=merged, x='NZ_SAC_AUD', y='AUS_CCont_AUD',
            scatter=False, color='black', line_kws={'linestyle': '--'})
plt.title('NZ SAC Rate vs AUS Commonwealth Contribution (both in AUD)')
plt.xlabel('NZ SAC Rate (AUD equivalent)')
plt.ylabel('AUS Commonwealth Contribution (AUD)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

r_all = merged['NZ_SAC_AUD'].corr(merged['AUS_CCont_AUD'])
print(f'Overall r(NZ_SAC, AUS_CCont) = {r_all:.3f}')

# Within-field correlation
within_r = (merged.groupby('field_of_study')
            .apply(lambda g: g['NZ_SAC_AUD'].corr(g['AUS_CCont_AUD']) if len(g) >= 3 else np.nan)
            .reset_index(name='r_NZ_vs_AUS')
            .sort_values('r_NZ_vs_AUS', ascending=False))
print('\nWithin-field correlation (NZ SAC vs AUS CCont):')
display(within_r)

# Heatmap of funding gap
gap_pivot = merged.pivot_table(index='field_of_study', columns='year', values='Fund_Gap_AUD')
plt.figure(figsize=(12, 6))
sns.heatmap(gap_pivot, annot=True, fmt='.0f', cmap='coolwarm', center=0)
plt.title('Funding Gap (AUS Commonwealth − NZ SAC, AUD) by Field and Year')
plt.tight_layout()
plt.show()

## 6. Distribution Comparison (Tukey Box Plots)

In [ ]:
# Overall distribution
plt.figure(figsize=(10, 5))
sns.boxplot(data=fund_long, x='Rate_AUD', y='Country', whis=1.5, palette='Set2')
sns.stripplot(data=fund_long, x='Rate_AUD', y='Country',
              palette='Set2', dodge=False, alpha=0.45, size=4)
plt.title('Government Funding Rate Distribution: NZ vs AUS (AUD)')
plt.xlabel('Funding Rate (AUD per EFTS / EFTSL)')
plt.tight_layout()
plt.show()

# By field of study
field_order = (fund_long.groupby('field_of_study')['Rate_AUD']
               .mean().sort_values(ascending=False).index.tolist())
plt.figure(figsize=(12, 7))
sns.boxplot(data=fund_long, x='Rate_AUD', y='field_of_study',
            hue='Country', order=field_order, whis=1.5)
sns.stripplot(data=fund_long, x='Rate_AUD', y='field_of_study',
              hue='Country', order=field_order, dodge=True, alpha=0.45, size=3,
              palette={'New Zealand': 'black', 'Australia': 'grey'})
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title='Country', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.title('Funding Rate Distribution by Field: NZ vs AUS (AUD)')
plt.xlabel('Funding Rate (AUD)')
plt.ylabel('Field of Study')
plt.tight_layout()
plt.show()

## 7. Additional Statistical Checks

Correlation, nonlinearity, and heterogeneity across fields.

In [ ]:
try:
    from scipy import stats
except ImportError:
    stats = None

if stats:
    r, p = stats.pearsonr(merged['NZ_SAC_AUD'], merged['AUS_CCont_AUD'])
    r_sp, p_sp = stats.spearmanr(merged['NZ_SAC_AUD'], merged['AUS_CCont_AUD'])
    print(f'Pearson r(NZ, AUS funding) = {r:.3f}  (p={p:.4f})')
    print(f'Spearman r = {r_sp:.3f}  (p={p_sp:.4f})')

    # Nonlinearity
    x = merged['NZ_SAC_AUD'].to_numpy(dtype=float)
    y = merged['AUS_CCont_AUD'].to_numpy(dtype=float)
    idx = np.argsort(x)
    x_s, y_s = x[idx], y[idx]

    def r2(yt, yp):
        return 1 - np.sum((yt - yp)**2) / np.sum((yt - yt.mean())**2)

    lin_r2  = r2(y_s, np.polyval(np.polyfit(x_s, y_s, 1), x_s))
    quad_r2 = r2(y_s, np.polyval(np.polyfit(x_s, y_s, 2), x_s))
    print(f'Linear R²: {lin_r2:.4f}  |  Quadratic R²: {quad_r2:.4f}')

    plt.figure(figsize=(9, 6))
    plt.scatter(x_s, y_s, color='black', alpha=0.55, label='Observed')
    plt.plot(x_s, np.polyval(np.polyfit(x_s, y_s, 1), x_s), color='steelblue', label='Linear fit')
    plt.plot(x_s, np.polyval(np.polyfit(x_s, y_s, 2), x_s), color='red', ls='--', label='Quadratic fit')
    plt.title('Nonlinearity Check: NZ vs AUS Funding Rates')
    plt.xlabel('NZ SAC Rate (AUD)')
    plt.ylabel('AUS Commonwealth Contribution (AUD)')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Heterogeneity across fields
    gap_groups = [g['Fund_Gap_AUD'].dropna().values for _, g in merged.groupby('field_of_study') if len(g) > 1]
    if len(gap_groups) >= 2:
        lev_stat, lev_p = stats.levene(*gap_groups, center='median')
        f_stat,   f_p   = stats.f_oneway(*gap_groups)
        kw_stat,  kw_p  = stats.kruskal(*gap_groups)
        print(f"\nLevene's test on funding gap: F={lev_stat:.3f}, p={lev_p:.4g}")
        print(f'ANOVA on funding gap: F={f_stat:.3f}, p={f_p:.4g}')
        print(f'Kruskal-Wallis: H={kw_stat:.3f}, p={kw_p:.4g}')

## Data Characteristics & First-Order Effects

**Variables:** NZ TEC SAC rate (NZD per EFTS, Level 2, excl. GST) and AUS `AvgCommonwealthContribution` (AUD per EFTSL, excl. GST). Both measure the per-student government subsidy paid to the institution. Neither includes the student-paid tuition fee.

**JRG discontinuity:** The 2021 JRG restructured AUS Commonwealth Contributions — substantially raising the subsidy for STEM and health (priority disciplines) and cutting it sharply for humanities, commerce, and law. NZ SAC rates had no equivalent shock; they follow smooth TEC review cycles with inflation-aligned increments.

**Currency conversion caveat:** AUD/NZD varies (~0.88–0.94 over 2019–2025). A fixed 0.91 rate is used throughout. Sensitivity to this assumption is limited because the key comparison is **growth rates** (indexed change), not absolute levels.

**Interpretation of funding gap:** A large `Fund_Gap_AUD > 0` means AUS funds that field more generously than NZ (in AUD equivalent). A post-2021 widening gap in STEM fields and a narrowing (or negative) gap in arts/commerce would confirm the JRG mechanism is active in the data.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid')

# Self-contained reload
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
raw_nz   = next((r / 'data' / 'raw' / 'NZ Funding' for r in candidate_roots
                 if (r / 'data' / 'raw' / 'NZ Funding').exists()), None)
clean    = next((r / 'data' / 'clean' for r in candidate_roots
                 if (r / 'data' / 'clean').exists()), None)

sac  = pd.read_csv(raw_nz / 'NZ_TEC_SAC_rates_all_years.csv')
aus  = pd.read_csv(clean  / 'AnnualFundingAUS2019-2026_category_summary.csv')

NZ_CAT_TO_KEY = {'V':1,'C':2,'N':3,'H':5,'B':6,'I':7,'J':8,'A':9}
NZD_TO_AUD = 0.91

nz_m = sac[sac['category_code'].isin(NZ_CAT_TO_KEY)].copy()
nz_m['category_key'] = nz_m['category_code'].map(NZ_CAT_TO_KEY)
nz_k = (nz_m.groupby(['category_key','year'], as_index=False)['sac_rate_level2_excl_gst']
        .mean().rename(columns={'sac_rate_level2_excl_gst': 'NZ_SAC_NZD'}))
nz_k['NZ_SAC_AUD'] = (nz_k['NZ_SAC_NZD'] * NZD_TO_AUD).round(0)

aus_c = (aus[aus['CategoryKey'] != 99]
         .rename(columns={'Year':'year','CategoryKey':'category_key',
                           'Category':'field_of_study',
                           'AvgCommonwealthContribution':'AUS_CCont_AUD'})
         [['year','category_key','field_of_study','AUS_CCont_AUD']])

common_yrs = sorted(set(nz_k['year']) & set(aus_c['year']))
m = (nz_k[nz_k['year'].isin(common_yrs)]
     .merge(aus_c[aus_c['year'].isin(common_yrs)], on=['category_key','year'], how='inner')
     .sort_values(['category_key','year']))
m['Fund_Gap_AUD'] = m['AUS_CCont_AUD'] - m['NZ_SAC_AUD']

print(f'=== NZ vs AUS Funding — Variable Summary ===')
print(f'Merged: {m.shape}  |  years: {common_yrs}')
r, p = stats.pearsonr(m['NZ_SAC_AUD'], m['AUS_CCont_AUD'])
print(f'Overall r(NZ, AUS) = {r:.3f}  (p={p:.4f})')

sk_nz  = stats.skew(m['NZ_SAC_AUD'])
sk_aus = stats.skew(m['AUS_CCont_AUD'])
print(f'Skewness NZ={sk_nz:.3f}  |  AUS={sk_aus:.3f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NZ vs AUS Funding — Data Characteristics & First-Order Effects',
             fontsize=12, fontweight='bold')

# Per-field r bar chart
cat_r = m.groupby('field_of_study').apply(
    lambda g: g['NZ_SAC_AUD'].corr(g['AUS_CCont_AUD']) if len(g) >= 3 else np.nan
).dropna().sort_values()
colors = ['#e74c3c' if v < 0 else '#27ae60' for v in cat_r]
axes[0].barh(cat_r.index, cat_r.values, color=colors)
axes[0].axvline(r, color='navy', ls='--', lw=2, label=f'Aggregate r={r:.2f}')
axes[0].set_title('A. Per-field NZ–AUS funding correlation')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='y', labelsize=8)

# Indexed change 2019=100 (aggregate)
BASE = 2019
agg_nz  = m.groupby('year')['NZ_SAC_AUD'].mean()
agg_aus = m.groupby('year')['AUS_CCont_AUD'].mean()
base_nz  = agg_nz[BASE]
base_aus = agg_aus[BASE]
axes[1].plot(agg_nz.index, agg_nz/base_nz*100, marker='o', label='New Zealand')
axes[1].plot(agg_aus.index, agg_aus/base_aus*100, marker='s', ls='--', label='Australia')
axes[1].axvline(2021, color='red', ls='--', lw=1.5, label='JRG 2021')
axes[1].axhline(100, color='grey', ls=':', lw=1)
axes[1].set_title(f'B. Avg funding indexed ({BASE}=100)')
axes[1].set_ylabel('Index')
axes[1].legend(fontsize=8)

# Distributions
axes[2].hist(m['NZ_SAC_AUD'],  bins=20, color='steelblue',  edgecolor='white', alpha=0.7, label='NZ (AUD)')
axes[2].hist(m['AUS_CCont_AUD'], bins=20, color='darkorange', edgecolor='white', alpha=0.6, label='AUS (AUD)')
axes[2].set_xlabel('Funding Rate (AUD)')
axes[2].set_title('C. Overlapping distributions')
axes[2].legend()

plt.tight_layout()
plt.show()

### What Is Learned

1. **Variable characteristics:** TODO — describe the NZ SAC rate distribution and AUS Commonwealth Contribution structure after running.
2. **Funding level comparison:** TODO — are AUS contributions higher or lower than NZ SAC rates (in AUD)? Does this vary by field?
3. **Parallel pre-trends (funding):** TODO — do NZ and AUS government funding rates grow at similar rates before 2021? Report per-field correlation of YoY growth rates.
4. **JRG discontinuity:** TODO — quantify the 2021 AUS shift relative to the NZ baseline using the field-level DiD table in Section 4. Which fields show the largest post-2021 divergence?
5. **Implications:** TODO — if NZ and AUS funding was broadly parallel pre-2021 and AUS diverged sharply post-2021, the funding channel is confirmed. Fields with the largest `DiD_AUD` are the most informative for the enrolment regression analysis.